# Steam ID Matching for Epic Games Free Games

This notebook matches Epic Games free game names to their Steam App IDs.

**Strategy:**
1. Load and clean Epic Games CSV (strip `(repeat)` suffixes, etc.)
2. Query `store.steampowered.com/api/storesearch` for each unique game name — results are cached to `raw_data/steam_search_cache.json` so re-runs are instant
3. Score each result with `rapidfuzz.WRatio` against the query name
4. High-confidence top result (score ≥ threshold) → single match
5. Lower-confidence → top N candidates kept for manual review
6. Save JSON: `{ epic_name: [[appid, steam_name], ...], ... }`

After running, copy `raw_data/steam_matches.json` to `processed_data/steam_matches_manual.json` and manually resolve any entries with multiple candidates by keeping only the correct `[appid, name]` pair.

In [3]:
import os
import re
import json
import time
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv
from rapidfuzz import fuzz

load_dotenv(Path("../.env.local") if Path("../.env.local").exists() else ".env.local")

STEAM_API_KEY = os.getenv("STEAM_API_KEY")
print(f"API key loaded: {'yes' if STEAM_API_KEY else 'NO - check .env.local'}")

API key loaded: yes


## 1. Load and clean Epic Games data

In [4]:
DATA_DIR = Path("../raw_data")
epic_csv = DATA_DIR / "epic games data.csv"

df = pd.read_csv(epic_csv)
print(f"Loaded {len(df)} rows")
df.head()

Loaded 541 rows


,serial no.,Game,Start,End,Genre
0,1,Subnautica,12-12-2018,27-12-2018,First-ever Epic free game.
1,2,Super Meat Boy,28-12-2018,10-01-2019,Second giveaway.
2,3,What Remains of Edith Finch,11-01-2019,24-01-2019,Early weekly rotation.
3,4,The Jackbox Party Pack,24-01-2019,07-02-2019,Party game bundle.
4,5,Axiom Verge,07-02-2019,21-02-2019,Metroidvania title.


In [5]:
def clean_game_name(name: str) -> str:
    """Strip trailing (repeat), (Repeat), whitespace, and normalise dashes."""
    name = str(name).strip()
    # Remove trailing (repeat) / (repeat, New Year slot) / etc.
    name = re.sub(r'\s*\(repeat[^)]*\)\s*$', '', name, flags=re.IGNORECASE).strip()
    return name

df["clean_name"] = df["Game"].apply(clean_game_name)

# Show any names that changed
changed = df[df["Game"] != df["clean_name"]][["Game", "clean_name"]]
print(f"{len(changed)} names cleaned:")
print(changed.to_string(index=False))

77 names cleaned:
                                                        Game                                          clean_name
                                            Celeste (repeat)                                             Celeste
                                Hyper Light Drifter (repeat)                                 Hyper Light Drifter
                                        Overcooked! (repeat)                                         Overcooked!
                                  Enter the Gungeon (repeat)                                   Enter the Gungeon
                                    Into the Breach (repeat)                                     Into the Breach
                                               ABZÛ (repeat)                                                ABZÛ
                                  Kingdom New Lands (repeat)                                   Kingdom New Lands
                                   Metro 2033 Redux (repeat)                  

## 2. Search Steam Store for each game (results cached)

In [7]:
SEARCH_CACHE_FILE = DATA_DIR / "steam_search_cache.json"
N_CANDIDATES = 5   # top results to fetch per query
RATE_LIMIT_S = 0.4  # seconds between requests

def steam_store_search(query: str, n: int = N_CANDIDATES) -> list[dict]:
    """
    Query the Steam store search endpoint.
    Returns a list of {id, name} dicts (up to n results).
    """
    url = "https://store.steampowered.com/api/storesearch/"
    params = {"term": query, "l": "english", "cc": "US"}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return [{"id": item["id"], "name": item["name"]} for item in items[:n]]

# Load existing cache (allows resuming if interrupted)
if SEARCH_CACHE_FILE.exists():
    with open(SEARCH_CACHE_FILE, "r", encoding="utf-8") as f:
        search_cache: dict[str, list[dict]] = json.load(f)
    print(f"Loaded {len(search_cache)} cached queries.")
else:
    search_cache = {}

unique_names = df["clean_name"].unique().tolist()
to_fetch = [n for n in unique_names if n not in search_cache]
print(f"{len(unique_names)} unique game names | {len(to_fetch)} need fetching (~{len(to_fetch)*RATE_LIMIT_S:.0f}s)")

for i, name in enumerate(to_fetch):
    try:
        search_cache[name] = steam_store_search(name)
    except Exception as e:
        print(f"  ERROR {name!r}: {e}")
        search_cache[name] = []
    if (i + 1) % 25 == 0 or (i + 1) == len(to_fetch):
        # Save incrementally so a crash doesn't lose progress
        with open(SEARCH_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(search_cache, f, indent=2, ensure_ascii=False)
        print(f"  {i+1}/{len(to_fetch)} fetched, cache saved.")
    time.sleep(RATE_LIMIT_S)

print("All queries cached.")

478 unique game names | 478 need fetching (~191s)
  25/478 fetched, cache saved.
  50/478 fetched, cache saved.
  75/478 fetched, cache saved.
  100/478 fetched, cache saved.
  125/478 fetched, cache saved.
  150/478 fetched, cache saved.
  175/478 fetched, cache saved.
  200/478 fetched, cache saved.
  225/478 fetched, cache saved.
  250/478 fetched, cache saved.
  275/478 fetched, cache saved.
  300/478 fetched, cache saved.
  325/478 fetched, cache saved.
  350/478 fetched, cache saved.
  375/478 fetched, cache saved.
  400/478 fetched, cache saved.
  425/478 fetched, cache saved.
  450/478 fetched, cache saved.
  475/478 fetched, cache saved.
  478/478 fetched, cache saved.
All queries cached.


## 3. Score and classify matches

Steam's search returns results ranked by its own relevance. We use `rapidfuzz.WRatio` to score the **top result** against the query:

- **Score ≥ HIGH_THRESHOLD** → single confident match
- **Score < HIGH_THRESHOLD** → keep top `N_CANDIDATES` for manual review
- **No results returned** → empty list (not on Steam)

In [8]:
HIGH_THRESHOLD = 88  # WRatio score above which we trust the top result as definitive

def classify_matches(epic_name: str) -> list[tuple[int, str, float]]:
    """
    Score Steam search results for `epic_name` using rapidfuzz.WRatio.
    Returns [(appid, steam_name, score), ...]:
      - single entry  → confident match
      - multiple      → needs manual review
      - empty         → not found on Steam
    """
    candidates = search_cache.get(epic_name, [])
    if not candidates:
        return []

    scored = [
        (c["id"], c["name"], fuzz.WRatio(epic_name, c["name"]))
        for c in candidates
    ]
    scored.sort(key=lambda x: x[2], reverse=True)

    best_appid, best_name, best_score = scored[0]
    if best_score >= HIGH_THRESHOLD:
        return [(best_appid, best_name, best_score)]
    else:
        return scored  # all candidates for manual review

match_results: dict[str, list[tuple[int, str, float]]] = {
    name: classify_matches(name) for name in unique_names
}

single_match = {k: v for k, v in match_results.items() if len(v) == 1}
multi_match  = {k: v for k, v in match_results.items() if len(v) > 1}
no_match     = {k: v for k, v in match_results.items() if len(v) == 0}

print(f"High-confidence single match : {len(single_match)}")
print(f"Multiple candidates (review) : {len(multi_match)}")
print(f"No match found               : {len(no_match)}")

High-confidence single match : 413
Multiple candidates (review) : 14
No match found               : 51


## 4. Review match quality

In [9]:
single_match   = {k: v for k, v in match_results.items() if len(v) == 1}
multi_match    = {k: v for k, v in match_results.items() if len(v) > 1}
no_match       = {k: v for k, v in match_results.items() if len(v) == 0}

print(f"High-confidence single match : {len(single_match)}")
print(f"Multiple candidates (review) : {len(multi_match)}")
print(f"No match found               : {len(no_match)}")

if no_match:
    print("\nUnmatched games:")
    for name in no_match:
        print(f"  - {name}")

High-confidence single match : 413
Multiple candidates (review) : 14
No match found               : 51

Unmatched games:
  - TowerFall Ascension
  - Tormentor x Punisher
  - Borderlands: The Handsome Collection
  - Stranger Things 3: The Game
  - 3 out of 10: Season One (Ep 1)
  - Hitman (2016)
  - Shadowrun Collection
  - RollerCoaster Tycoon 3: Complete Edition
  - Pillars of Eternity: Definitive Edition
  - Tyranny: Gold Edition
  - Star Wars Battlefront II: Celebration Edition
  - 3 out of 10: Season Two
  - Idle Champions of the Forgotten Realms (pack)
  - NBA 2K21
  - Rogue Company Season 4 Epic Pack (DLC)
  - KID A MNESIA EXHIBITION
  - Antstream – Epic Welcome Pack
  - theHunter: Call of the Wild
  - Godfall: Challenger Edition
  - Second Extinction
  - Tomb Raider Trilogy (Tomb Raider GOTY, Rise of the Tomb Raider, Shadow of the Tomb Raider)
  - Bioshock: The Collection
  - Shop Titans (bundle)
  - Rumbleverse Boom Boxer Pack (DLC)
  - Realm Royale Reforged Epic Launch Bundle 

In [10]:
# Show all multi-candidate games for review
print("Games needing manual review:\n")
for epic_name, candidates in sorted(multi_match.items()):
    print(f"{epic_name!r}")
    for appid, steam_name, score in candidates:
        print(f"    {appid:>8}  {score:5.1f}  {steam_name}")
    print()

Games needing manual review:

'Art of Rally'
     1421640   85.5  art of rally 4k wallpaper pack
     2250420   85.5  art of rally: australia
      550320   83.3  art of rally
     1297600   71.4  art of rally OST

'Death Stranding'
     3280350   24.5  DEATH STRANDING 2: ON THE BEACH
     1850570   24.5  DEATH STRANDING DIRECTOR'S CUT
     1946740   24.5  DEATH STRANDING Digital Artbook

'Demon’s Tilt'
     1008730   31.3  DEMON'S TILT Collectors Pack
      422510   25.0  DEMON'S TILT

'For Honor'
     1366900   50.0  FOR HONOR™ - Warmonger Hero
     4312460   40.0  Juren - Hero - FOR HONOR
     1837260   40.0  FOR HONOR™ - Pirate Hero
     1641940   40.0  FOR HONOR™ - Kyoshin Hero
      304390   31.6  FOR HONOR™

'Gladius – Relics of War (Warhammer 40,000)'
      489630   71.0  Warhammer 40,000: Gladius - Relics of War
      870530   63.3  Warhammer 40,000: Gladius - Relics of War - Soundtrack
      870550   63.3  Warhammer 40,000: Gladius - Relics of War - Wallpapers

'Hot Wheels Un

## 5. Print unmatched games

In [11]:
if no_match:
    print("Games with no Steam search results (likely F2P, delisted, or Epic-exclusive):\n")
    for name in sorted(no_match):
        print(f"  - {name}")
else:
    print("All games returned at least one Steam search result.")

Games with no Steam search results (likely F2P, delisted, or Epic-exclusive):

  - 3 out of 10: Season One (Ep 1)
  - 3 out of 10: Season Two
  - Additional minor DLC / cosmetics slots (holiday pack group)*
  - Albion Online – Rogue Journeyman Bundle
  - Antstream – Epic Welcome Pack
  - Bioshock: The Collection
  - Borderlands: The Handsome Collection
  - Dark and Darker – Legendary Status (DLC)
  - Destiny 2: Legacy Collection (DLC bundle)
  - Divine Knockout
  - Dragon Age: Inquisition – Game of the Year Edition
  - Evil Dead: The Game
  - Fallout: New Vegas – Ultimate Edition
  - Firestone: Online Idle RPG – Epic Pack (DLC)
  - Freshly Frosted + Rumble Club pack
  - Godfall: Challenger Edition
  - Hell Let Loose (repeat, final Holiday 2024)
  - Hitman (2016)
  - Idle Champions of the Forgotten Realms (DLC pack)
  - Idle Champions of the Forgotten Realms (pack)
  - KID A MNESIA EXHIBITION
  - Kingdom Come: Deliverance (repeat, New Year slot)
  - LISA: The Joyful – Definitive Edition

## 6. Save output JSON

Format: `{ "Epic Game Name": [[appid, steam_name], ...] }`

- Single-element lists = high-confidence match
- Multi-element lists = review needed, pick the correct one and delete the rest
- Empty list = not found on Steam (game may be free-to-play only, delisted, or Epic-exclusive)

In [12]:
output: dict[str, list[list]] = {}

for epic_name, candidates in match_results.items():
    output[epic_name] = [[appid, steam_name] for appid, steam_name, *_ in candidates]

OUT_FILE = DATA_DIR / "steam_matches.json"
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output)} entries to {OUT_FILE}")

# Summary
single = sum(1 for v in output.values() if len(v) == 1)
multi  = sum(1 for v in output.values() if len(v) > 1)
empty  = sum(1 for v in output.values() if len(v) == 0)
print(f"  Single match (done)   : {single}")
print(f"  Multiple candidates   : {multi}  ← needs manual review")
print(f"  No match              : {empty}  ← not found on Steam")

Saved 478 entries to ../raw_data/steam_matches.json
  Single match (done)   : 413
  Multiple candidates   : 14  ← needs manual review
  No match              : 51  ← not found on Steam
